In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [ ]:
file_path=rf"Inputfile.xlsx"

In [ ]:
workbook=openpyxl.load_workbook(file_path)
sheetname=workbook.sheetnames
print(sheetname)


In [4]:
cols=['MAKE & MODEL', 'ENGINE', 'REGULAR PLUG', 'STK#', 'GAP']
df_s = pd.DataFrame(columns=cols)
df_s

,MAKE & MODEL,ENGINE,REGULAR PLUG,STK#,GAP


In [ ]:
for s in sheetname:
    sno=int(s.split(" ")[1])
    if sno%2==1:
        # print("The sheet is odd numbered:", sno)
        #print(s)
        df1=pd.read_excel(file_path,sheet_name=s, skiprows=1, usecols='A:E')
        df2=pd.read_excel(file_path,sheet_name=s, skiprows=1,usecols="G:K")
        df1["Sheet Number"] = sno
        df2["Sheet Number"] = sno
        df2.rename(columns={'MAKE & MODEL.1':"MAKE & MODEL", 'ENGINE.1':"ENGINE", 'REGULAR PLUG.1':"REGULAR PLUG", 'STK#.1':"STK#", 'GAP.1':"GAP"}, inplace=True)
        df_s=pd.concat([df_s,df1,df2])#.dropna(axis=0, how='all', inplace=True)
        print(sno, df_s.columns)
    else:
        # print("The sheet is even numbered:", sno)
        #print(s)    
        df3=pd.read_excel(file_path,sheet_name=s,usecols='A:E')
        df4=pd.read_excel(file_path,sheet_name=s,usecols="G:K")
        df3["Sheet Number"] = sno
        df4["Sheet Number"] = sno
        df4.rename(columns={'MAKE & MODEL.1':"MAKE & MODEL", 'ENGINE.1':"ENGINE", 'REGULAR PLUG.1':"REGULAR PLUG", 'STK#.1':"STK#", 'GAP.1':"GAP"}, inplace=True)   
        df_s=pd.concat([df_s,df3,df4])
        print(sno, df_s.columns)

In [6]:
df_s.columns

Index(['MAKE & MODEL', 'ENGINE', 'REGULAR PLUG', 'STK#', 'GAP',
       'Sheet Number'],
      dtype='object')

In [ ]:
df_s

In [8]:
df_s.dropna(axis=0, how='all', inplace=True)
df_s.reset_index(drop=True, inplace=True)


In [9]:
df_s["Make"]=np.where(
    (df_s["ENGINE"].isna()) & 
    (df_s["REGULAR PLUG"].isna()) & 
    (df_s["STK#"].isna()) & 
    (df_s["GAP"].isna()), 
    df_s["MAKE & MODEL"], 
    np.nan
)

In [10]:
df_s["Model"]=np.where( 
    df_s["Make"]==df_s["MAKE & MODEL"],
    np.nan,
    df_s["MAKE & MODEL"])


In [13]:
df_cleaned=df_s[["Sheet Number",'Make','Model', 'ENGINE', 'REGULAR PLUG', 'STK#', 'GAP']]
df_cleaned['Make'] = df_cleaned['Make'].ffill()
# df_cleaned['Model'] = df_cleaned['Model'].ffill()
df_cleaned["Remove"]=np.where(
    (df_s["ENGINE"].isna()) & 
    (df_s["REGULAR PLUG"].isna()) & 
    (df_s["STK#"].isna()) & 
    (df_s["GAP"].isna()) &
    (df_s["Model"].isna()), 
    0,
    1
)

In [ ]:
df_cleaned = df_cleaned[df_cleaned['Remove'].apply(str).str.contains("1", regex=False, na=False, case=False)]
df_cleaned